🔹 What is a Callback in LangChain?

A callback is a hook that lets you listen to events happening inside a LangChain run.
Think of it like a logging/monitoring system where you can track:

When a chain, tool, or LLM call starts and ends

What inputs/outputs were passed

How much time it took

How many tokens were consumed (useful for OpenAI costs)

Custom behaviors (e.g., sending logs to Slack, saving traces in DB, or visualizing execution flow)

In LangChain, callbacks are handled by CallbackHandler classes. You can use built-in ones or write your own.

🔹 Types of Callbacks in LangChain

Built-in Handlers

StdOutCallbackHandler → prints logs to console

StreamingStdOutCallbackHandler → streams LLM responses as they are generated

LangChainTracer → sends traces to LangSmith dashboard

AsyncCallbackHandler → async version for async chains

Custom Handlers

Extend BaseCallbackHandler and override event methods.

🔹 Key Callback Events

Some important methods you can override:

on_llm_start → when an LLM is called

on_llm_end → after LLM responds

on_chain_start / on_chain_end → chain lifecycle

on_tool_start / on_tool_end → tool usage

on_text → when LLM streams tokens

on_error → when something goes wrong

🔹 Example 1: Simple logging with StdOutCallbackHandler

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.callbacks import StdOutCallbackHandler
from dotenv import load_dotenv
load_dotenv()

# Define the model
load_dotenv()

# Create callback handler
handler = StdOutCallbackHandler()

# Pass it into LLM
llm = ChatOpenAI(model="gpt-4o-mini", callbacks=[handler], verbose=True)

# Run LLM
response = llm.invoke("Write a short poem about callbacks in LangChain.")
print("\nFinal Response:", response.content)



Final Response: In the realm of code where logic flows,  
LangChain weaves, as each result glows.  
A whisper of functions, a dance in the night,  
Callbacks emerge, bringing visions to light.  

With promise in hand, they listen and wait,  
Gathering data to orchestrate fate.  
As tokens align, and responses entwine,  
A symphony plays, each note a design.  

From intentions to actions, connections unfold,  
In the heart of the chain, stories are told.  
So here’s to the callbacks, the unsung delight,  
In the world of LangChain, they sparkle so bright.  


🔹 Example 2: Streaming tokens as they generate

In [2]:
from langchain_openai import ChatOpenAI
from langchain.callbacks.streaming_stdout import StreamingStdOutCallbackHandler

llm = ChatOpenAI(
    model="gpt-4o-mini",
    streaming=True, 
    callbacks=[StreamingStdOutCallbackHandler()]
)

# The response is streamed token by token
resp = llm.invoke("Explain LangChain callbacks in one sentence.")


LangChain callbacks are mechanisms that allow users to hook into the execution flow of a LangChain application, enabling custom actions or responses at various stages of processing tasks.

🔹 Example 3: Custom Callback Handler

In [3]:
import time
from langchain.callbacks.base import BaseCallbackHandler
from langchain_openai import ChatOpenAI

class MyCustomHandler(BaseCallbackHandler):
    def on_llm_start(self, serialized, prompts, **kwargs):
        self.start_time = time.time()
        print(f"\n[LLM START] Prompt: {prompts}")

    def on_llm_end(self, response, **kwargs):
        duration = time.time() - self.start_time
        print(f"[LLM END] Response: {response.generations[0][0].text}")
        print(f"[TIME TAKEN] {duration:.2f} sec")

    def on_chain_start(self, serialized, inputs, **kwargs):
        print(f"\n[CHAIN START] Inputs: {inputs}")

    def on_chain_end(self, outputs, **kwargs):
        print(f"[CHAIN END] Outputs: {outputs}\n")

# Attach to LLM
llm = ChatOpenAI(model="gpt-4o-mini", callbacks=[MyCustomHandler()], verbose=True)

resp = llm.invoke("Summarize the importance of callbacks in LangChain.")
print("Final:", resp)



[LLM START] Prompt: ['Human: Summarize the importance of callbacks in LangChain.']
[LLM END] Response: Callbacks in LangChain are essential for enhancing the functionality and user experience of applications that leverage language models. They serve several important purposes:

1. **Monitoring and Logging**: Callbacks help developers monitor the execution of various components, providing insights into performance, execution flow, and potential issues.

2. **Customization**: They allow for the customization of behavior during processing, enabling the modification of input and output data, as well as the ability to implement additional logic at various stages of the execution.

3. **Error Handling**: Callbacks provide a structured way to handle errors and exceptions that may arise during the execution of language model tasks, facilitating robust application behavior.

4. **Event Handling**: They enable the implementation of event-driven architectures, where specific actions can be trigg

🔹 Example 4: Callback with a Chain

In [4]:
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from langchain_openai import ChatOpenAI
from langchain.callbacks import StdOutCallbackHandler

# Handler
handler = StdOutCallbackHandler()

# Prompt + Chain
prompt = PromptTemplate.from_template("Translate the following to French: {sentence}")
llm = ChatOpenAI(model="gpt-4o-mini", callbacks=[handler], verbose=True)
chain = LLMChain(llm=llm, prompt=prompt, callbacks=[handler])

chain.run("Callbacks are very useful in LangChain.")


C:\Users\anilk\AppData\Local\Temp\ipykernel_7568\2327488798.py:12: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  chain = LLMChain(llm=llm, prompt=prompt, callbacks=[handler])
C:\Users\anilk\AppData\Local\Temp\ipykernel_7568\2327488798.py:14: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  chain.run("Callbacks are very useful in LangChain.")




> Entering new LLMChain chain...
Prompt after formatting:
Translate the following to French: Callbacks are very useful in LangChain.

> Finished chain.


'Les rappels sont très utiles dans LangChain.'

🔹 Real-world Use Cases for Callbacks

Monitoring & Debugging → log inputs/outputs, detect failures

Token Usage Tracking → track costs on OpenAI/Azure/GCP

Streaming → live chatbot UIs, token-by-token responses

Tracing & Visualization → send traces to LangSmith or your own dashboard

Custom Integrations → e.g., send error alerts to Slack/Teams, log to a database, or push execution details into Prometheus/Grafana

✅ Summary:
Callbacks = Event listeners for LangChain runs.
They let you debug, monitor, stream, and extend chain/tool/LLM executions.

🔹 Callbacks with Agents + Tools

Agents in LangChain decide which tools (e.g., calculator, search, DB) to call.
Callbacks can log which tool was chosen, what inputs it received, and what outputs it returned.

✅ Example 1: Agent with Callback Logging

In [7]:
from langchain.agents import initialize_agent, AgentType, Tool
from langchain_openai import ChatOpenAI
from langchain.callbacks.base import BaseCallbackHandler
import time

# ---- Custom Callback Handler ----
class AgentLogger(BaseCallbackHandler):

    def on_tool_start(self, serialized, input_str, **kwargs):
        name = serialized.get("name") if serialized else "UnknownTool"
        print(f"\n🛠️ [TOOL START] {name} with input: {input_str}")
        self.tool_start = time.time()

    def on_tool_end(self, output, **kwargs):
        duration = time.time() - getattr(self, "tool_start", time.time())
        print(f"✅ [TOOL END] Output: {output} (took {duration:.2f}s)")

    def on_agent_action(self, action, **kwargs):
        print(f"🤖 [AGENT DECISION] Action: {action.tool}, Input: {action.tool_input}")

    def on_chain_start(self, serialized, inputs, **kwargs):
        name = serialized.get("name") if serialized else "UnknownChain"
        print(f"\n🔗 [CHAIN START] {name} Inputs: {inputs}")

    def on_chain_end(self, outputs, **kwargs):
        print(f"🔗 [CHAIN END] Outputs: {outputs}")

# ---- Define a simple tool ----
def multiply(a: int, b: int) -> str:
    return str(int(a) * int(b))

tools = [
    Tool(
        name="Multiplier",
        func=lambda q: multiply(*map(int, q.split())),
        description="multiply two numbers given as 'a b'"
    )
]

# ---- LLM + Agent with Callback ----
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    callbacks=[AgentLogger()],
    verbose=True
)

# ---- Run Agent ----
agent.run("multiply 3 4")



🔗 [CHAIN START] UnknownChain Inputs: {'input': 'multiply 3 4'}


> Entering new AgentExecutor chain...
🤖 [AGENT DECISION] Action: Multiplier, Input: 3 4
To find the product of 3 and 4, I will use the multiplication tool. 
Action: Multiplier
Action Input: 3 4
Observation: 12
Thought:I now know the final answer
Final Answer: 12
🔗 [CHAIN END] Outputs: {'output': '12'}

> Finished chain.


'12'

✅ Example 2: Streaming Responses + Tool Tracking

In [6]:
from langchain.callbacks.streaming_stdout import StreamingStdOutCallbackHandler

llm = ChatOpenAI(
    model="gpt-4o-mini",
    streaming=True,
    callbacks=[StreamingStdOutCallbackHandler(), AgentLogger()]  # mix multiple callbacks
)

agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

agent.run("Multiply 21 and 5, then explain the result in one sentence.")




> Entering new AgentExecutor chain...
To find the result of multiplying 21 and 5, I will perform the multiplication operation.  
Action: Multiplier  
Action Input: 21 5  To find the result of multiplying 21 and 5, I will perform the multiplication operation.  
Action: Multiplier  
Action Input: 21 5  
Observation: 105
Thought:I now know the final answer. The product of 21 and 5 is 105, which represents the total when 21 is added to itself 5 times or when 5 is added to itself 21 times.  
Final Answer: The result of multiplying 21 and 5 is 105, meaning that if you add 21 together 5 times, you get 105.I now know the final answer. The product of 21 and 5 is 105, which represents the total when 21 is added to itself 5 times or when 5 is added to itself 21 times.  
Final Answer: The result of multiplying 21 and 5 is 105, meaning that if you add 21 together 5 times, you get 105.

> Finished chain.


'The result of multiplying 21 and 5 is 105, meaning that if you add 21 together 5 times, you get 105.'

🔹 Real-World Uses of Callbacks in Agents

Debugging → See which tool the agent picked, why, and with what inputs

Monitoring → Track latency and errors for each tool call

Observability dashboards → Send traces to LangSmith, Prometheus, or custom DB

Compliance logging → Keep a full audit trail of tool usage

UI Streaming → Show partial responses while tool calls are running

✅ Summary:
Callbacks with Agents + Tools let you trace decision-making and tool execution step by step. You can combine them with streaming and cost tracking for a production-grade observability stack.

In [9]:
! pip install langchain langchain-openai langsmith python-dotenv

Defaulting to user installation because normal site-packages is not writeable
  Using cached tiktoken-0.12.0-cp314-cp314-win_amd64.whl.metadata (6.9 kB)
   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   --------------------------- ------------ 0.8/1.1 MB 4.4 MB/s eta 0:00:01
   ---------------------------------------- 1.1/1.1 MB 4.2 MB/s  0:00:00
Using cached tiktoken-0.12.0-cp314-cp314-win_amd64.whl (921 kB)

   ------ --------------------------------- 1/6 [regex]
  Attempting uninstall: openai
   ------ --------------------------------- 1/6 [regex]
    Found existing installation: openai 2.6.0
   ------ --------------------------------- 1/6 [regex]
   -------------------- ------------------- 3/6 [openai]
   -------------------- ------------------- 3/6 [openai]
   -------------------- ------------------- 3/6 [openai]
   -------------------- ------------------- 3/6 [openai]
    Uninstalling openai-2.6.0:
   -------------------- ------------------- 3/6 [openai]
   


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: C:\Python314\python.exe -m pip install --upgrade pip


In [13]:
from langchain.agents import initialize_agent, AgentType, Tool
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
import re
from langsmith import Client
from langchain.callbacks.tracers import LangChainTracer

client = Client()
tracer = LangChainTracer(project_name="agent-tracing-demo-1")

# Load environment variables
load_dotenv()

# ---- Define Tool ----
def multiply_tool(query: str) -> str:
    numbers = list(map(int, re.findall(r"\d+", query)))
    if len(numbers) < 2:
        return "Please provide at least two numbers"
    return str(numbers[0] * numbers[1])

tools = [
    Tool(
        name="Multiplier",
        func=multiply_tool,
        description="Multiply two numbers from input text"
    )
]

# ---- Initialize LLM ----
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

# ---- Initialize Agent ----
agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    callbacks=[tracer],
    verbose=True
)

# ---- Run Agent ----
response = agent.run("What is 12 multiplied by 8?")
print("\nFinal Answer:", response)



> Entering new AgentExecutor chain...
To find the answer, I need to multiply 12 by 8. 
Action: Multiplier
Action Input: "12 multiplied by 8"
Observation: 96
Thought:I now know the final answer
Final Answer: 96

> Finished chain.

Final Answer: 96
